In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [2]:
class EquationState(TypedDict):
    a:int
    b:int
    c:int

    equation:str
    discriminant:int
    result:str

In [3]:
from typing_extensions import Literal


def equation(state:EquationState):
    a = state['a']
    b = state['b']
    c = state['c']

    state['equation'] = f"{a}x^2 + {b}x + {c} = 0"
    
    return {"equation": state['equation']}


    

def cal_discriminant(state:EquationState):
    a = state['a']
    b = state['b']
    c = state['c']

    state['discriminant'] = b**2 - 4*a*c

    return {"discriminant": state['discriminant']}



def no_real_roots(state: EquationState):
    # D < 0 → complex roots
    a = state["a"]
    b = state["b"]
    D = state["discriminant"]

    real_part = -b / (2 * a)
    imag_part = (abs(D) ** 0.5) / (2 * a)

    state["result"] = (
        f"No real roots. Complex roots are: "
        f"{real_part} + {imag_part}i and {real_part} - {imag_part}i"
    )
    
    return {"result": state["result"]}


def real_roots(state: EquationState):
    # D > 0 → two real roots
    a = state["a"]
    b = state["b"]
    D = state["discriminant"]

    sqrtD = D ** 0.5

    x1 = (-b + sqrtD) / (2 * a)
    x2 = (-b - sqrtD) / (2 * a)

    state["result"] = f"Two distinct real roots: x1 = {x1}, x2 = {x2}"

    return {"result": state["result"]}


def repeated_roots(state: EquationState):
    # D = 0 → one repeated root
    a = state["a"]
    b = state["b"]

    x = -b / (2 * a)

    state["result"] = f"One repeated real root: x = {x}"

    return {"result": state["result"]}

def check_condition(state:EquationState) -> Literal["no_real_roots","real_roots","repeated_roots"]:
    D = state['discriminant']
    if D < 0:
        return "no_real_roots"
    elif D > 0:
        return "real_roots"
    else:
        return "repeated_roots"
    




In [5]:
graph=StateGraph(EquationState)

graph.add_node("equation",equation)
graph.add_node("cal_discriminant",cal_discriminant)
graph.add_node("no_real_roots",no_real_roots)
graph.add_node("real_roots",real_roots)
graph.add_node("repeated_roots",repeated_roots)

graph.add_edge(START,"equation")
graph.add_edge("equation","cal_discriminant")
graph.add_conditional_edges("cal_discriminant",check_condition)
graph.add_edge("no_real_roots",END)
graph.add_edge("real_roots",END)
graph.add_edge("repeated_roots",END)


workflow=graph.compile()


In [6]:
initial_state = {
    "a": 1,
    "b": -3,
    "c": 2
}

workflow.invoke(initial_state)

{'a': 1,
 'b': -3,
 'c': 2,
 'equation': '1x^2 + -3x + 2 = 0',
 'discriminant': 1,
 'result': 'Two distinct real roots: x1 = 2.0, x2 = 1.0'}

In [7]:
# repated roots
initial_state = {
    "a": 1,
    "b": -2,
    "c": 1
}

workflow.invoke(initial_state)

{'a': 1,
 'b': -2,
 'c': 1,
 'equation': '1x^2 + -2x + 1 = 0',
 'discriminant': 0,
 'result': 'One repeated real root: x = 1.0'}

In [8]:
# no real roots
initial_state = {
    "a": 1,
    "b": 2,
    "c": 5
}
workflow.invoke(initial_state)

{'a': 1,
 'b': 2,
 'c': 5,
 'equation': '1x^2 + 2x + 5 = 0',
 'discriminant': -16,
 'result': 'No real roots. Complex roots are: -1.0 + 2.0i and -1.0 - 2.0i'}